XGBoost Model

In [1]:
AGluon_target="GOALS"

In [110]:
import pandas as pd
from sklearn.svm import SVR

#!pip uninstall -y scikit-learn
#!pip install scikit-learn==1.3.2
#!pip install --upgrade u8darts
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from autogluon.timeseries.splitter import ExpandingWindowSplitter
from sklearn.ensemble import RandomForestRegressor
import torch
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

from autogluon.tabular import TabularPredictor
import xgboost as xgb

df=pd.read_csv("testML3.csv").iloc[:,1:]
max_t=df['time'].max()
print(max_t)
names= df['name'].unique()
print(names)
time_df=pd.DataFrame()
for i in range(len(names)):
    name=names[i]
    first_filtered= df[df['name'] == name]
    unique_teamvals= first_filtered['Team'].unique()
    for t in range(len(unique_teamvals)):
        team=unique_teamvals[t]
        new_filtered= first_filtered[first_filtered['Team'] == team]
        times=[]
        filtered = new_filtered[new_filtered["minutes"] > 0]
        for g in range(len(filtered)):
            times.append(max_t-g)
        times.reverse()
        filtered["time"]=times
        if(len(unique_teamvals)>1):
            filtered['name']=filtered['name'].values[0]+str(t)
        time_df=pd.concat([time_df, filtered], axis=0, ignore_index=True)

time_df.to_csv("ML_training2.csv")
if(AGluon_target=="GOALS"):
    features=["opposition_xgc","Own_Attacking_form","XG_slope","rolling_shots",
              "minutes","rolling_Threat","rolling_XG_historic","Rolling_adjusted_XG2"]
    target="expected_goals"
elif(AGluon_target=="Assist"):
    features=["name", "expected_assists","opposition_xgc",
               "Own_Attacking_form","Rolling_creativity", "Rolling_adjusted_XA2","Cluster",
               "rolling_XA_historic","minutes","rolling_key_passes","XA_slope"]
    target="expected_assists"


df = pd.read_csv("ML_training2.csv").iloc[:, 1:]
names = ['Mohamed_Salah','Kai_Havertz1','Ollie_Watkins','Antoine_Semenyo','Bryan_Mbeumo',
         'João_Pedro Junqueira de Jesus','Danny_Welbeck','Nicolas_Jackson','Jean-Philippe_Mateta',
         'Dominic_Calvert-Lewin','Diogo_Teixeira da Silva','Erling_Haaland','Alexander_Isak',
         'Chris_Wood0','Matheus_Santos Carneiro Da Cunha','Dominic_Solanke',
         'Gabriel_dos Santos Magalhães','William_Saliba','Lucas_Digne','Ezri_Konsa Ngoyo',
         'Lewis_Dunk','Levi_Colwill0','Antonee_Robinson','Trent_Alexander-Arnold',
         'Andrew_Robertson','Joško_Gvardiol','Rico_Lewis','Diogo_Dalot Teixeira',
         'Dan_Burn','Pedro_Porro','Rayan_Aït-Nouri','Kai_Havertz0','Gabriel_Martinelli Silva',
         'Bukayo_Saka','Martin_Ødegaard','Morgan_Rogers','Antoine_Semenyo','Marcus_Tavernier',
         'Bryan_Mbeumo','Noni_Madueke','Cole_Palmer0','Eberechi_Eze','Dwight_McNeil',
         'Diogo_Teixeira da Silva','Luis_Díaz','Mohamed_Salah','Phil_Foden',
         'Bruno_Borges Fernandes','Harvey_Barnes0','Anthony_Gordon0',
         'Morgan_Gibbs-White','Brennan_Johnson0','Dejan_Kulusevski','James_Maddison1','Jarrod_Bowen']
errors = []
df=df[df['position'].isin(["FWD", "DEF", "MID"])]

df["opposition_xg"] = df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
df["opposition_xgc"] = df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
# If not forecasting (is_pred == 0), exclude season 30
player_df = df[df["season"] != 30].copy()
player_df=player_df[player_df['time']>10]
#player_df = player_df[player_df["name"].isin(names)]
max_time=player_df["time"].max()

train_df_2 = player_df[player_df["time"] <= max_time-15]  

train_df = train_df_2[features].copy()
train_df=train_df.fillna(0)
train_y=train_df_2[target].copy()
model_xg = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.05, max_depth=5,min_child_weight=6)
model_xg=SVR(kernel='rbf', C=0.5, epsilon=0.1,gamma=0.1)
#model_xg = RandomForestRegressor(n_estimators=100)


model_xg.fit(train_df,train_y)
# Get feature importances


#predictor = TabularPredictor(label=target, problem_type='regression',path=f"Gluon_{AGluon_target}")
#predictor.fit(train_df, presets='medium_quality')


113
['Fábio_Ferreira Vieira' 'Gabriel_Fernando de Jesus'
 'Gabriel_dos Santos Magalhães' 'Kai_Havertz' 'Jurriën_Timber'
 'Jorge_Luiz Frello Filho' 'Jakub_Kiwior' 'Gabriel_Martinelli Silva'
 'Ethan_Nwaneri' 'Martin_Ødegaard' 'David_Raya Martin' 'Declan_Rice'
 'Bukayo_Saka' 'William_Saliba' 'Thomas_Partey' 'Kieran_Tierney'
 'Leandro_Trossard' 'Benjamin_White' 'Oleksandr_Zinchenko'
 'Raheem_Sterling' 'Riccardo_Calafiori' 'Myles_Lewis-Skelly'
 'Mikel_Merino' 'Leon_Bailey' 'Ross_Barkley' 'Emiliano_Buendía Stati'
 'Matty_Cash' 'Leander_Dendoncker' 'Moussa_Diaby'
 'Diego_Carlos Santos Silva' 'Lucas_Digne' 'Jhon_Durán' 'Boubacar_Kamara'
 'Ezri_Konsa Ngoyo' 'Ian_Maatsen' 'Emiliano_Martínez Romero' 'John_McGinn'
 'Tyrone_Mings' 'Kosta_Nedeljković' 'Robin_Olsen' 'Pau_Torres'
 'Jacob_Ramsey' 'Morgan_Rogers' 'Youri_Tielemans' 'Ollie_Watkins'
 'Amadou_Onana' 'Jaden_Philogene' 'Lamare_Bogarde' 'Max_Aarons'
 'Tyler_Adams' 'Jaidon_Anthony' 'David_Brooks' 'Ryan_Christie'
 'Lewis_Cook' 'Enes_Ünal' 'Hamed

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_17512\1040980617.py:69: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df["opposition_xg"] = df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_17512\1040980617.py:70: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df["opposition_xgc"] = df.apply(lambda row: row[23] if row[18] else row[21], axis=1)


SVR(C=0.5, gamma=0.1)

In [112]:
importances = model_xg.feature_importances_
features = train_df.columns

# Create a DataFrame for easy viewing
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feature_importance_df)
feature_importance_df.head(10).plot(kind='barh', x='Feature', y='Importance', legend=False)
plt.gca().invert_yaxis()  # Most important at the top
plt.title("Top 10 Feature Importances - RandomForest")
plt.tight_layout()
plt.show()

AttributeError: 'SVR' object has no attribute 'feature_importances_'

In [111]:

names1=['Mohamed_Salah','Kai_Havertz1','Ollie_Watkins','Antoine_Semenyo','Bryan_Mbeumo',
         'João_Pedro Junqueira de Jesus','Danny_Welbeck','Nicolas_Jackson','Jean-Philippe_Mateta',
         'Dominic_Calvert-Lewin','Diogo_Teixeira da Silva','Erling_Haaland','Alexander_Isak',
         'Chris_Wood0','Matheus_Santos Carneiro Da Cunha','Dominic_Solanke',
         'Gabriel_dos Santos Magalhães','William_Saliba','Lucas_Digne','Ezri_Konsa Ngoyo',
         'Lewis_Dunk','Levi_Colwill0','Antonee_Robinson','Trent_Alexander-Arnold',
         'Andrew_Robertson','Joško_Gvardiol','Rico_Lewis','Diogo_Dalot Teixeira',
         'Dan_Burn','Pedro_Porro','Rayan_Aït-Nouri','Kai_Havertz0','Gabriel_Martinelli Silva',
         'Bukayo_Saka','Martin_Ødegaard','Morgan_Rogers','Antoine_Semenyo','Marcus_Tavernier',
         'Bryan_Mbeumo','Noni_Madueke','Cole_Palmer0','Eberechi_Eze','Dwight_McNeil',
         'Diogo_Teixeira da Silva','Luis_Díaz','Mohamed_Salah','Phil_Foden',
         'Bruno_Borges Fernandes','Harvey_Barnes0','Anthony_Gordon0',
         'Morgan_Gibbs-White','Brennan_Johnson0','Dejan_Kulusevski','James_Maddison1','Jarrod_Bowen']
errors=[]
for i in range(len(names1)):
    name=names1[i]
    print(name)
    player_df = df[df["season"] != 30].copy()
    player_df.sort_values(by='time', inplace=True)

    test_df = player_df[player_df["name"] == name] 

    val_series=test_df[features].copy().iloc[-8:,:]

    actuals=test_df[[target]].copy().iloc[-8:,:]
    print(actuals)
    pred=model_xg.predict(val_series)
    print(pred)
    actuals=actuals.values

    mse = mean_squared_error(actuals, pred)
    errors.append(mse)
    print("Running average RMSE:", sum(errors)/len(errors))


#0.0818-SVR
#0.0823

#0.074-XGB
#0.0744(0.069)
#0.076

#RF-0.074



Mohamed_Salah
       expected_goals
16776            1.96
16777            0.07
16778            0.44
16779            0.09
16780            1.09
16781            0.21
16782            0.09
16783            0.00
[0.66127278 0.68902373 0.55472192 0.54270988 0.63681717 0.65744547
 0.56937364 0.52274738]
Running average RMSE: 0.39957952303999844
Kai_Havertz1
     expected_goals
298            1.01
299            0.20
300            0.26
301            0.32
302            0.00
303            0.20
304            0.40
305            0.00
[0.28243119 0.13478654 0.09745845 0.20256565 0.14967885 0.20123695
 0.20057731 0.12472059]
Running average RMSE: 0.2405116160918175
Ollie_Watkins
      expected_goals
3101            0.56
3102            0.00
3103            0.00
3104            0.28
3105            0.44
3106            0.00
3107            0.21
3108            0.36
[0.38961277 0.10733093 0.17950229 0.15201898 0.22908887 0.09281544
 0.37300022 0.31744944]
Running average RMSE: 0.167450651687

TS Mixer

In [48]:
Ts_Mixer_target="GOALS"

In [42]:
#!pip install --upgrade u8darts
from darts.models import TSMixerModel
from darts.dataprocessing.transformers import Scaler
from darts.timeseries import TimeSeries

import pandas as pd

if(Ts_Mixer_target=="GOALS"):
    past_cov=["shots", "Threat"]
    future_cov=["minutes", "Own_Attacking_form", "opposition_xgc","Cluster"]
    input_chunk=8
    hidden_size=20
    hidden_size=16
    random_state=36
    ff_size=16
    ff_size=32
    num_blocks=1
    target="expected_goals"
elif(Ts_Mixer_target=="Assist"):
    past_cov=["key_passes", "creativity"]
    future_cov=["minutes", "Own_Attacking_form", "opposition_xgc","Cluster"]
    input_chunk=10
    hidden_size=20
    random_state=40
    ff_size=32
    num_blocks=1
    target="expected_assists"
    
    
    

# Load your data and define the list of players
df = pd.read_csv("ML_training2.csv").iloc[:, 1:]
names = ['Mohamed_Salah','Kai_Havertz1','Ollie_Watkins','Antoine_Semenyo','Bryan_Mbeumo',
         'João_Pedro Junqueira de Jesus','Danny_Welbeck','Nicolas_Jackson','Jean-Philippe_Mateta',
         'Dominic_Calvert-Lewin','Diogo_Teixeira da Silva','Erling_Haaland','Alexander_Isak',
         'Chris_Wood0','Matheus_Santos Carneiro Da Cunha','Dominic_Solanke',
         'Gabriel_dos Santos Magalhães','William_Saliba','Lucas_Digne','Ezri_Konsa Ngoyo',
         'Lewis_Dunk','Levi_Colwill0','Antonee_Robinson','Trent_Alexander-Arnold',
         'Andrew_Robertson','Joško_Gvardiol','Rico_Lewis','Diogo_Dalot Teixeira',
         'Dan_Burn','Pedro_Porro','Rayan_Aït-Nouri','Kai_Havertz0','Gabriel_Martinelli Silva',
         'Bukayo_Saka','Martin_Ødegaard','Morgan_Rogers','Antoine_Semenyo','Marcus_Tavernier',
         'Bryan_Mbeumo','Noni_Madueke','Cole_Palmer0','Eberechi_Eze','Dwight_McNeil',
         'Diogo_Teixeira da Silva','Luis_Díaz','Mohamed_Salah','Phil_Foden',
         'Bruno_Borges Fernandes','Harvey_Barnes0','Anthony_Gordon0',
         'Morgan_Gibbs-White','Brennan_Johnson0','Dejan_Kulusevski','Jarrod_Bowen']
df = df[df["season"] != 30]
name_counts = df['name'].value_counts()
print(name_counts)
names_more_than_15 = name_counts[name_counts > 30].index.tolist()

name_to_index = {name: idx for idx, name in enumerate(names_more_than_15)}
player_df = df[df["name"].isin(names_more_than_15)]
player_df = player_df[player_df["season"] != 30]
player_df["opposition_xg"] = player_df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
player_df["opposition_xgc"] = player_df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
player_df['expected_goals'] = player_df['expected_goals'].clip(upper=1.5)
player_df.fillna(0, inplace=True)
player_df["name_index"] = player_df["name"].map(name_to_index)

max_time=player_df['time'].max()
print(max_time)

train_df=player_df[player_df["time"]<=max_time-12]

train_target = TimeSeries.from_group_dataframe(train_df, time_col="time", group_cols="name_index",value_cols=target)

train_past_cov = TimeSeries.from_group_dataframe(train_df, time_col="time", group_cols="name_index",value_cols=past_cov)

train_future_cov = TimeSeries.from_group_dataframe(train_df, time_col="time", group_cols="name_index",value_cols=future_cov)
Scaler1=Scaler()

past_cov_scaled=Scaler1.fit_transform(train_past_cov)

# Define model parameters
input_chunk = input_chunk         # number of past time steps used by the model
output_chunk = 8        # forecast horizon
training_length = input_chunk + output_chunk

model=TSMixerModel(
        input_chunk_length=input_chunk, 
        output_chunk_length=output_chunk, 
        hidden_size=hidden_size, 
        ff_size=ff_size,
        num_blocks=num_blocks,
        use_static_covariates=False,
        use_reversible_instance_norm=True,
        activation ="LeakyReLU",
        random_state=random_state)
model.fit(train_target, past_covariates =past_cov_scaled, future_covariates =train_future_cov,epochs=15)

model.save(f"TS_{Ts_Mixer_target}.pkl")
    

ImportError: cannot import name '_check_method_params' from 'sklearn.utils.validation' (C:\Users\OleJacobSimensen\OneDrive - twoday\Documents\Python_test\venv\Lib\site-packages\sklearn\utils\validation.py)

In [ ]:
Test

In [54]:
from sklearn.metrics import mean_squared_error

pred=0

df = pd.read_csv("ML_training2.csv").iloc[:, 1:]
names = ['Mohamed_Salah','Kai_Havertz1','Ollie_Watkins','Antoine_Semenyo','Bryan_Mbeumo',
         'João_Pedro Junqueira de Jesus','Danny_Welbeck','Nicolas_Jackson','Jean-Philippe_Mateta',
         'Dominic_Calvert-Lewin','Diogo_Teixeira da Silva','Erling_Haaland','Alexander_Isak',
         'Chris_Wood0','Matheus_Santos Carneiro Da Cunha','Dominic_Solanke',
         'Gabriel_dos Santos Magalhães','William_Saliba','Lucas_Digne','Ezri_Konsa Ngoyo',
         'Lewis_Dunk','Levi_Colwill0','Antonee_Robinson','Trent_Alexander-Arnold',
         'Andrew_Robertson','Joško_Gvardiol','Rico_Lewis','Diogo_Dalot Teixeira',
         'Dan_Burn','Pedro_Porro','Rayan_Aït-Nouri','Kai_Havertz0','Gabriel_Martinelli Silva',
         'Bukayo_Saka','Martin_Ødegaard','Morgan_Rogers','Antoine_Semenyo','Marcus_Tavernier',
         'Bryan_Mbeumo','Noni_Madueke','Cole_Palmer0','Eberechi_Eze','Dwight_McNeil',
         'Diogo_Teixeira da Silva','Luis_Díaz','Mohamed_Salah','Phil_Foden',
         'Bruno_Borges Fernandes','Harvey_Barnes0','Anthony_Gordon0',
         'Morgan_Gibbs-White','Brennan_Johnson0','Dejan_Kulusevski','Jarrod_Bowen']
player_df=df.copy()
namelist=df["name"].unique()
name_to_index = {name: idx for idx, name in enumerate(namelist)}
player_df["opposition_xg"] = player_df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
player_df["opposition_xgc"] = player_df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
player_df['expected_goals'] = player_df['expected_goals'].clip(upper=1.5)
player_df["name_index"] = player_df["name"].map(name_to_index)

max_ind=player_df["time"].max()
offset=0
print(max_ind)
players_to_use=namelist
if(pred==0):
    print('dddddddddddddddddddddddddddddddddddddd')
    player_df = player_df[player_df["season"] != 30]
    offset=1
    players_to_use=names
errors=[]

pred_df=pd.DataFrame()
for j in range(len(players_to_use)):
    player_preds=[]
    print(players_to_use[j])
    
    new_df=player_df[player_df["name"]==players_to_use[j]]
    
    player_preds.append(players_to_use[j])
    test_df=new_df[new_df["season"]==25]
    if(len(test_df)<1):
        continue

    if(len(new_df)<=(output_chunk+input_chunk)):
        ind_list=list(range(new_df["time"].min()-1, max_ind-output_chunk-11-8*offset, -1))
        dummy_df=pd.DataFrame()
        dummy_df["time"]=ind_list
        dummy_df["name_index"]=new_df["name_index"].values[0]
        new_df=pd.concat([new_df, dummy_df], ignore_index=True)
        new_df.sort_values(by='time', inplace=True)

    new_df.fillna(0, inplace=True)
    test_target = TimeSeries.from_group_dataframe(new_df, time_col="time", group_cols="name_index",value_cols=target)

    test_past_cov = TimeSeries.from_group_dataframe(new_df, time_col="time", group_cols="name_index",value_cols=past_cov)[0][:-output_chunk]

    test_future_cov = TimeSeries.from_group_dataframe(new_df, time_col="time", group_cols="name_index",value_cols=future_cov)[0]


    test_past_cov_scaled=Scaler1.transform(test_past_cov)

    
    loaded_model = TSMixerModel.load(f"TS_{Ts_Mixer_target}.pkl")

    forecast = loaded_model.predict(
                            n=output_chunk,
                            series=test_target[0][:-output_chunk],
                            past_covariates=test_past_cov_scaled,
                            future_covariates=test_future_cov,
    )
    #forecast_inv = scaler_target.inverse_transform(forecast)

    actual = test_target[0][-output_chunk:]
    
    print("Actual:")
    print(actual)
    print("Forecast:")
    print(forecast)
    
    # Convert TimeSeries to numpy arrays using .values()
    mse = mean_squared_error(actual.values(), forecast.values())
    for t in range(len(forecast.values())):
        player_preds.append(forecast.values()[t][0])
    player_preds.append(new_df["position"].values[-1])
    append_df=pd.DataFrame([player_preds], columns=["Name", "p1","p2","p3","p4","p5","p6","p7","p8","position"])
    pred_df = pd.concat([pred_df, append_df], ignore_index=True)

    print(append_df)
    errors.append(mse)
    print(sum(errors)/len(errors))
pred_df.to_csv(f"TS_{Ts_Mixer_target}.csv")
#0.0825
#0.021

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\327595042.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  player_df["opposition_xg"] = player_df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\327595042.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  player_df["opposition_xgc"] = player_df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\327595042.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the cave

114
dddddddddddddddddddddddddddddddddddddd
Mohamed_Salah


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Predicting: |                                                                            | 0/? [00:00<?, ?it/s…

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\327595042.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df.fillna(0, inplace=True)
Only 1 TimeSeries (lists) were provided which is lower than the number of series (n=394) used to fit Scaler. This can result in a mismatch between the series and the underlying transformers.


...

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\327595042.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df.fillna(0, inplace=True)
Only 1 TimeSeries (lists) were provided which is lower than the number of series (n=394) used to fit Scaler. This can result in a mismatch between the series and the underlying transformers.


Actual:
<TimeSeries (DataArray) (time: 8, component: 1, sample: 1)> Size: 64B
array([[[0.03]],

       [[0.27]],

       [[0.11]],

       [[0.07]],

       [[0.27]],

       [[0.33]],

       [[0.05]],

       [[0.  ]]])
Coordinates:
  * time       (time) int64 64B 99 100 101 102 103 104 105 106
  * component  (component) <U14 56B 'expected_goals'
Dimensions without coordinates: sample
Attributes:
    static_covariates:  static_covariates  name_index\ncomponent             ...
    hierarchy:          None
    metadata:           None
Forecast:
<TimeSeries (DataArray) (time: 8, component: 1, sample: 1)> Size: 64B
array([[[0.13532088]],

       [[0.1184963 ]],

       [[0.13354191]],

       [[0.17182615]],

       [[0.18249609]],

       [[0.16419797]],

       [[0.20539492]],

       [[0.06296214]]])
Coordinates:
  * time       (time) int64 64B 99 100 101 102 103 104 105 106
  * component  (component) object 8B 'expected_goals'
Dimensions without coordinates: sample
Attributes:
    st

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Predicting: |                                                                            | 0/? [00:00<?, ?it/s…

Actual:
<TimeSeries (DataArray) (time: 8, component: 1, sample: 1)> Size: 64B
array([[[0.02]],

       [[0.2 ]],

       [[0.  ]],

       [[0.92]],

       [[0.08]],

       [[0.12]],

       [[0.24]],

       [[0.18]]])
Coordinates:
  * time       (time) int64 64B 99 100 101 102 103 104 105 106
  * component  (component) <U14 56B 'expected_goals'
Dimensions without coordinates: sample
Attributes:
    static_covariates:  static_covariates  name_index\ncomponent             ...
    hierarchy:          None
    metadata:           None
Forecast:
<TimeSeries (DataArray) (time: 8, component: 1, sample: 1)> Size: 64B
array([[[0.19200372]],

       [[0.26125916]],

       [[0.25841627]],

       [[0.21423084]],

       [[0.30862872]],

       [[0.28182886]],

       [[0.2631226 ]],

       [[0.2876519 ]]])
Coordinates:
  * time       (time) int64 64B 99 100 101 102 103 104 105 106
  * component  (component) object 8B 'expected_goals'
Dimensions without coordinates: sample
Attributes:
    st

In [ ]:
LSTM


In [24]:
LSTM_target="GOALS"

In [43]:
#!pip install --upgrade u8darts
from darts.models import TFTModel
from darts.dataprocessing.transformers import Scaler
from darts.timeseries import TimeSeries
import torch.nn as nn
import torch



import pandas as pd
class RMSELoss(nn.Module):
    def __init__(self):
        super(RMSELoss, self).__init__()
        self.mse = nn.MSELoss()
    def forward(self, yhat, y):
        return torch.sqrt(self.mse(yhat, y))
        
if(LSTM_target=="GOALS"):
    past_cov=["shots", "Threat","Rolling_adjusted_XG2"]
    future_cov=["minutes", "Own_Attacking_form", "opposition_xgc","Cluster"]
    input_chunk=8
    hidden_size=20
    lstm_layers=2
    num_attention_heads=1
    dropout=0.1
    hidden_continuous_size=8
    random_state=36
    target="expected_goals"
elif(LSTM_target=="Assist"):
    past_cov=["key_passes", "creativity"]
    future_cov=["minutes", "Own_Attacking_form", "opposition_xgc","Cluster"]
    input_chunk=8
    hidden_size=64
    lstm_layers=2
    num_attention_heads=2
    dropout=0.1
    hidden_continuous_size=4
    target="expected_assists"
    
    
    

# Load your data and define the list of players
df = pd.read_csv("ML_training2.csv").iloc[:, 1:]
names = ['Mohamed_Salah','Kai_Havertz1','Ollie_Watkins','Antoine_Semenyo','Bryan_Mbeumo',
         'João_Pedro Junqueira de Jesus','Danny_Welbeck','Nicolas_Jackson','Jean-Philippe_Mateta',
         'Dominic_Calvert-Lewin','Diogo_Teixeira da Silva','Erling_Haaland','Alexander_Isak',
         'Chris_Wood0','Matheus_Santos Carneiro Da Cunha','Dominic_Solanke',
         'Gabriel_dos Santos Magalhães','William_Saliba','Lucas_Digne','Ezri_Konsa Ngoyo',
         'Lewis_Dunk','Levi_Colwill0','Antonee_Robinson','Trent_Alexander-Arnold',
         'Andrew_Robertson','Joško_Gvardiol','Rico_Lewis','Diogo_Dalot Teixeira',
         'Dan_Burn','Pedro_Porro','Rayan_Aït-Nouri','Kai_Havertz0','Gabriel_Martinelli Silva',
         'Bukayo_Saka','Martin_Ødegaard','Morgan_Rogers','Antoine_Semenyo','Marcus_Tavernier',
         'Bryan_Mbeumo','Noni_Madueke','Cole_Palmer0','Eberechi_Eze','Dwight_McNeil',
         'Diogo_Teixeira da Silva','Luis_Díaz','Mohamed_Salah','Phil_Foden',
         'Bruno_Borges Fernandes','Harvey_Barnes0','Anthony_Gordon0',
         'Morgan_Gibbs-White','Brennan_Johnson0','Dejan_Kulusevski','Jarrod_Bowen']
df = df[df["season"] != 30]
name_counts = df['name'].value_counts()
print(name_counts)
names_more_than_15 = name_counts[name_counts > 30].index.tolist()

name_to_index = {name: idx for idx, name in enumerate(names_more_than_15)}
player_df = df[df["name"].isin(names_more_than_15)]
player_df = player_df[player_df["season"] != 30]
player_df["opposition_xg"] = player_df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
player_df["opposition_xgc"] = player_df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
player_df['expected_goals'] = player_df['expected_goals'].clip(upper=1.5)
player_df.fillna(0, inplace=True)
player_df["name_index"] = player_df["name"].map(name_to_index)

max_time=player_df['time'].max()
print(max_time)

train_df=player_df[player_df["time"]<=max_time-12]

train_target = TimeSeries.from_group_dataframe(train_df, time_col="time", group_cols="name_index",value_cols=target)

train_past_cov = TimeSeries.from_group_dataframe(train_df, time_col="time", group_cols="name_index",value_cols=past_cov)

train_future_cov = TimeSeries.from_group_dataframe(train_df, time_col="time", group_cols="name_index",value_cols=future_cov)
Scaler1=Scaler()

past_cov_scaled=Scaler1.fit_transform(train_past_cov)

# Define model parameters
input_chunk = input_chunk         # number of past time steps used by the model
output_chunk = 8        # forecast horizon
training_length = input_chunk + output_chunk

model = TFTModel(
    input_chunk_length=input_chunk,
    output_chunk_length=output_chunk,
    hidden_size=hidden_size,
    lstm_layers=lstm_layers,
    num_attention_heads=num_attention_heads,
    dropout=dropout,
    batch_size=32,
    hidden_continuous_size=hidden_continuous_size,
    n_epochs=15,
    force_reset=True,
    use_static_covariates=False,
    loss_fn=RMSELoss()
)
model.fit(train_target, past_covariates =past_cov_scaled, future_covariates =train_future_cov)

model.save(f"LSTM_{LSTM_target}.pkl")
    

ImportError: cannot import name '_check_method_params' from 'sklearn.utils.validation' (C:\Users\OleJacobSimensen\OneDrive - twoday\Documents\Python_test\venv\Lib\site-packages\sklearn\utils\validation.py)

In [47]:
from sklearn.metrics import mean_squared_error

pred=1

df = pd.read_csv("ML_training2.csv").iloc[:, 1:]
names = ['Mohamed_Salah','Kai_Havertz1','Ollie_Watkins','Antoine_Semenyo','Bryan_Mbeumo',
         'João_Pedro Junqueira de Jesus','Danny_Welbeck','Nicolas_Jackson','Jean-Philippe_Mateta',
         'Dominic_Calvert-Lewin','Diogo_Teixeira da Silva','Erling_Haaland','Alexander_Isak',
         'Chris_Wood0','Matheus_Santos Carneiro Da Cunha','Dominic_Solanke',
         'Gabriel_dos Santos Magalhães','William_Saliba','Lucas_Digne','Ezri_Konsa Ngoyo',
         'Lewis_Dunk','Levi_Colwill0','Antonee_Robinson','Trent_Alexander-Arnold',
         'Andrew_Robertson','Joško_Gvardiol','Rico_Lewis','Diogo_Dalot Teixeira',
         'Dan_Burn','Pedro_Porro','Rayan_Aït-Nouri','Kai_Havertz0','Gabriel_Martinelli Silva',
         'Bukayo_Saka','Martin_Ødegaard','Morgan_Rogers','Antoine_Semenyo','Marcus_Tavernier',
         'Bryan_Mbeumo','Noni_Madueke','Cole_Palmer0','Eberechi_Eze','Dwight_McNeil',
         'Diogo_Teixeira da Silva','Luis_Díaz','Mohamed_Salah','Phil_Foden',
         'Bruno_Borges Fernandes','Harvey_Barnes0','Anthony_Gordon0',
         'Morgan_Gibbs-White','Brennan_Johnson0','Dejan_Kulusevski','Jarrod_Bowen']
player_df=df.copy()
namelist=df["name"].unique()
name_to_index = {name: idx for idx, name in enumerate(namelist)}
player_df["opposition_xg"] = player_df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
player_df["opposition_xgc"] = player_df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
player_df['expected_goals'] = player_df['expected_goals'].clip(upper=1.5)
player_df["name_index"] = player_df["name"].map(name_to_index)

max_ind=player_df["time"].max()
offset=0
print(max_ind)
players_to_use=namelist
if(pred==0):
    print('dddddddddddddddddddddddddddddddddddddd')
    player_df = player_df[player_df["season"] != 30]
    offset=1
    players_to_use=names
errors=[]

pred_df=pd.DataFrame()
for j in range(len(players_to_use)):
    player_preds=[]
    print(players_to_use[j])
    
    new_df=player_df[player_df["name"]==players_to_use[j]]
    
    player_preds.append(players_to_use[j])
    test_df=new_df[new_df["season"]==25]
    if(len(test_df)<1):
        continue

    if(len(new_df)<=(output_chunk+input_chunk)):
        ind_list=list(range(new_df["time"].min()-1, max_ind-output_chunk-11-8*offset, -1))
        dummy_df=pd.DataFrame()
        dummy_df["time"]=ind_list
        dummy_df["name_index"]=new_df["name_index"].values[0]
        new_df=pd.concat([new_df, dummy_df], ignore_index=True)
        new_df.sort_values(by='time', inplace=True)

    new_df.fillna(0, inplace=True)
    test_target = TimeSeries.from_group_dataframe(new_df, time_col="time", group_cols="name_index",value_cols=target)

    test_past_cov = TimeSeries.from_group_dataframe(new_df, time_col="time", group_cols="name_index",value_cols=past_cov)[0][:-output_chunk]

    test_future_cov = TimeSeries.from_group_dataframe(new_df, time_col="time", group_cols="name_index",value_cols=future_cov)[0]


    test_past_cov_scaled=Scaler1.transform(test_past_cov)

    
    loaded_model = TFTModel.load(f"LSTM_{LSTM_target}.pkl")

    forecast = loaded_model.predict(
                            n=output_chunk,
                            series=test_target[0][:-output_chunk],
                            past_covariates=test_past_cov_scaled,
                            future_covariates=test_future_cov,
    )
    #forecast_inv = scaler_target.inverse_transform(forecast)

    actual = test_target[0][-output_chunk:]
    
    print("Actual:")
    print(actual)
    print("Forecast:")
    print(forecast)
    
    # Convert TimeSeries to numpy arrays using .values()
    mse = mean_squared_error(actual.values(), forecast.values())
    for t in range(len(forecast.values())):
        player_preds.append(forecast.values()[t][0])
    player_preds.append(new_df["position"].values[-1])
    append_df=pd.DataFrame([player_preds], columns=["Name", "p1","p2","p3","p4","p5","p6","p7","p8","position"])
    pred_df = pd.concat([pred_df, append_df], ignore_index=True)

    print(append_df)
    errors.append(mse)
    print(sum(errors)/len(errors))
pred_df.to_csv(f"TFT_{LSTM_target}.csv")
#0.087

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\1607768472.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  player_df["opposition_xg"] = player_df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\1607768472.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  player_df["opposition_xgc"] = player_df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\1607768472.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the c

114
Fábio_Ferreira Vieira
Gabriel_Fernando de Jesus


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Predicting: |                                                                            | 0/? [00:00<?, ?it/s…

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_7124\1607768472.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df.fillna(0, inplace=True)
Only 1 TimeSeries (lists) were provided which is lower than the number of series (n=394) used to fit Scaler. This can result in a mismatch between the series and the underlying transformers.


...

Only 1 TimeSeries (lists) were provided which is lower than the number of series (n=394) used to fit Scaler. This can result in a mismatch between the series and the underlying transformers.


Actual:
<TimeSeries (DataArray) (time: 8, component: 1, sample: 1)> Size: 64B
array([[[0.72]],

       [[0.  ]],

       [[0.04]],

       [[0.02]],

       [[0.03]],

       [[0.04]],

       [[0.02]],

       [[0.03]]])
Coordinates:
  * time       (time) int64 64B 107 108 109 110 111 112 113 114
  * component  (component) <U14 56B 'expected_goals'
Dimensions without coordinates: sample
Attributes:
    static_covariates:  static_covariates  name_index\ncomponent             ...
    hierarchy:          None
    metadata:           None
Forecast:
<TimeSeries (DataArray) (time: 8, component: 1, sample: 1)> Size: 64B
array([[[0.02871703]],

       [[0.01751466]],

       [[0.01740236]],

       [[0.00571614]],

       [[0.0282685 ]],

       [[0.0167055 ]],

       [[0.027473  ]],

       [[0.02700565]]])
Coordinates:
  * time       (time) int64 64B 107 108 109 110 111 112 113 114
  * component  (component) object 8B 'expected_goals'
Dimensions without coordinates: sample
Attributes:
    

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Predicting: |                                                                            | 0/? [00:00<?, ?it/s…

Actual:
<TimeSeries (DataArray) (time: 8, component: 1, sample: 1)> Size: 64B
array([[[0.6 ]],

       [[0.66]],

       [[0.27]],

       [[0.4 ]],

       [[0.  ]],

       [[0.7 ]],

       [[0.  ]],

       [[0.7 ]]])
Coordinates:
  * time       (time) int64 64B 107 108 109 110 111 112 113 114
  * component  (component) <U14 56B 'expected_goals'
Dimensions without coordinates: sample
Attributes:
    static_covariates:  static_covariates  name_index\ncomponent             ...
    hierarchy:          None
    metadata:           None
Forecast:
<TimeSeries (DataArray) (time: 8, component: 1, sample: 1)> Size: 64B
array([[[0.28536378]],

       [[0.27628441]],

       [[0.25411427]],

       [[0.67543303]],

       [[0.56352412]],

       [[0.31996328]],

       [[0.8341171 ]],

       [[0.29351244]]])
Coordinates:
  * time       (time) int64 64B 107 108 109 110 111 112 113 114
  * component  (component) object 8B 'expected_goals'
Dimensions without coordinates: sample
Attributes:
    

In [1]:
future = 5
column_list = []
column_list.append("Name")
for k in range(future):
    column_list.append(f"p{k+1}")
column_list.append("position")
print(column_list)

['Name', 'p1', 'p2', 'p3', 'p4', 'p5', 'position']
